In [1]:
import pandas as pd
import numpy as np
import librosa
import torch
import torch.nn as nn
import lime
import lime.lime_tabular

import matplotlib.pyplot as plt
import plotly.express as px

import huggingface_hub

import os
from dotenv import load_dotenv
import random
from tqdm import tqdm
import glob
from torch.utils.data import Dataset, DataLoader, random_split
from collections import OrderedDict
import soundfile as sf


from transformers import Wav2Vec2Model

from scipy.optimize import brentq
from scipy.interpolate import interp1d

from huggingface_hub import hf_hub_download

In [2]:
# load in huggingface authorization token
load_dotenv()
hf_auth=os.getenv("hf_auth")

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# import Wav2Vec Model from HuggingFace

w2v_model=Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

wav2vec_model_path=r'https://huggingface.co/rde6mn/no_aug_w2v_4s'
wav2vec_name='best_model.pth'
w2v_checkpoint=hf_hub_download(
    repo_id='rde6mn/no_aug_w2v_4s',
    filename=wav2vec_name
)
w2v_checkpoint=torch.load(w2v_checkpoint, map_location=device)
state_dict=w2v_checkpoint["model_state_dict"]
new_state_dict=OrderedDict()

# this checkpoint includes a 'wrapper' which places 'wav2vec.' in front of each layer name
# need to undo this to properly load the model

for k, v in state_dict.items():

    # keep only wav2vec backbone
    if k.startswith("wav2vec."):

        new_key = k.replace("wav2vec.", "")
        new_state_dict[new_key] = v


w2v_model.load_state_dict(new_state_dict)
w2v_model.eval()

# w2v parameters
SR = 16000
MAX_LEN = 4 * SR
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-5
NUM_WORKERS = 4

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
audiofile_dir=r"G:\My Drive\ASVSpoof_Data\unzipped2019\LA\LA\ASVspoof2019_LA_eval\flac"
audiofiles=glob.glob(os.path.join(audiofile_dir, "*.flac"))

def process_audio(audio_path):
    try:
        waveform, sr = sf.read(audio_path)
        if len(waveform.shape) > 1:
            waveform = waveform.mean(axis=1)

        if sr != SR:
            waveform = librosa.resample(
                waveform,
                orig_sr=sr,
                target_sr=SR
            )
    except Exception as e:
        try:
            waveform, sr = librosa.load(
                audio_path,
                sr=SR,
                mono=True
            )
        except Exception as e2:
            print(f"FAILED TO LOAD: {audio_path}")
            return None
    if len(waveform) > MAX_LEN:
        waveform = waveform[:MAX_LEN]
    else:
        waveform = np.pad(
            waveform,
            (0, MAX_LEN - len(waveform))
        )
    waveform = torch.tensor(
        waveform
    ).float()

    return waveform

In [22]:
waveforms=[]

# Preprocess the waveforms
for file in audiofiles:
    audio_path=file
    waveform=process_audio(audio_path)
    if waveform is not None:
        waveforms.append(waveform)
        if len(waveforms)==100:
                break
    else:
        print(f"FAILED TO LOAD: {audio_path}")
    


In [24]:
len(waveforms)

100

In [25]:
def generate_output(waveform):
    with torch.no_grad():
        outputs = w2v_model(waveform)
    return outputs



In [26]:
wav0_output=generate_output(waveforms[0])
wav0_output

RuntimeError: Given groups=1, weight of size [512, 1, 10], expected input[1, 64000, 1] to have 1 channels, but got 64000 channels instead